In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, make_scorer

In [2]:
# 前処理データの読み込み
train = pd.read_csv('train_pre.csv')

In [3]:
# 目的変数 '賃料' と説明変数を指定
X = train.drop(columns=['賃料'])  # 説明変数
y = train['賃料']  # 目的変数

In [4]:
# トレーニングデータとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# ランダムフォレスト回帰モデルの初期化
rf = RandomForestRegressor(random_state=42)

In [6]:
# ハイパーパラメータのグリッドを定義
param_grid = {
    'n_estimators': [10, 100, 300],
    'max_depth': [None, 10, 50, 100],
    'min_samples_split': [2, 10, 100],
    'min_samples_leaf': [1, 5, 9]
}

In [7]:
# RMSEスコアのカスタムスコア関数
def rmse_scorer(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [8]:
# グリッドサーチの実行
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, scoring=make_scorer(rmse_scorer, greater_is_better=False), cv=5, verbose=1, n_jobs=-1)

In [9]:
# グリッドサーチをトレーニングデータで実行
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


/home/whitesalena/python/pytorch/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 50, 100],
                         'min_samples_leaf': [1, 5, 9],
                         'min_samples_split': [2, 10, 100],
                         'n_estimators': [10, 100, 300]},
             scoring=make_scorer(rmse_scorer, greater_is_better=False, response_method='predict'),
             verbose=1)

In [10]:
# 最適なハイパーパラメータ
print("Best parameters found: ", grid_search.best_params_)

Best parameters found:  {'max_depth': None, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 100}


In [11]:
# 最適なモデル
best_model = grid_search.best_estimator_

In [12]:
# テストデータでの予測
y_pred = best_model.predict(X_test)

In [13]:
# テストデータでのRMSE評価
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {test_rmse}")

Test RMSE: 36083.80493425021
